In [43]:
import numpy as np
import pandas as pd
import argparse
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

from ct_rep.utils.sklearn_classifiers import knn_on_embeddings
from ct_rep.utils.pytorch_mlp import Classifier
from ct_rep.utils.utils import extract_baseline_embs, split_train_test
import os
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from ct_rep.celltype.celltype_pred import evaluate
from ct_rep.utils.utils import sample_data, adapt_vars
from ct_rep.utils.utils import normalize, split_train_test
from sklearn.decomposition import PCA


In [ ]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
results_path = "ct_rep/celltype/within_dataset_results.csv"

results_df = pd.read_csv(results_path)
columns=['config', 'tissue', 'dataset_ref', 'filename_ref', 'dataset_query', 'filename_query', 'classifier', 'method', 'emb_path', 'accuracy', 'f1_score_weighted', 'f1_score_macro']
assert results_df.columns.tolist() == columns
# results_df = pd.DataFrame(columns = columns)

In [ ]:
from ct_rep.celltype.celltype_pred import run

dataset_configs = [
    # ("neurips_2021_multiome", "adata_gex.h5ad", "batch", "s1", "cell_type", "neurips_2021_multiome", "adata_gex.h5ad", "batch", "s3", "cell_type", "blood"),
    # ("4e6932db-5a78-40e4-b961-f87f66ba139a", "adata.h5ad", "donor_id", "%", "cell_type", "4e6932db-5a78-40e4-b961-f87f66ba139a", "adata.h5ad", "donor_id", "%", "cell_type", "blood"),
    # ("70bcd657-48e6-405f-9c89-1de62cf16c3e", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "70bcd657-48e6-405f-9c89-1de62cf16c3e", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "retina"),
    # ("8e394b2a-4682-44de-969e-d3fdc5748eda", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "8e394b2a-4682-44de-969e-d3fdc5748eda", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "blood"),
    # ("d20c92a0-69f8-4612-838c-22f185878ea4", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "d20c92a0-69f8-4612-838c-22f185878ea4", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "blood"),
    # ("982952fb-dee0-4498-a072-3dc498b06aae", "adata.h5ad", "donor_id", "%", "cell_type", "982952fb-dee0-4498-a072-3dc498b06aae", "adata.h5ad", "donor_id", "%", "cell_type", "brain"),
    # ("a9dd3a9d-ef9f-49bb-b97b-e435817fc890", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "a9dd3a9d-ef9f-49bb-b97b-e435817fc890", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "lung"),
    # ("114bc07e-2d8a-45a9-9ca5-6c95137be44f", "adata.h5ad", "donor_id", "%", "cell_type", "114bc07e-2d8a-45a9-9ca5-6c95137be44f", "adata.h5ad", "donor_id", "%", "cell_type", "salivary glands"),
    # ("7840398e-2e12-49d2-b21a-f59a1908057e", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "7840398e-2e12-49d2-b21a-f59a1908057e", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "skeletal muscle"),
    # ("f7be56a5-55fc-4beb-8d5d-ffec21c11906", "adata.h5ad", "donor_id", "%", "cell_type", "f7be56a5-55fc-4beb-8d5d-ffec21c11906", "adata.h5ad", "donor_id", "%", "cell_type", "lung"),
    # ("74892429-375e-4a08-a6a0-716c8d53f078", "adata.h5ad", "donor_id", "%", "cell_type", "74892429-375e-4a08-a6a0-716c8d53f078", "adata.h5ad", "donor_id", "%", "cell_type", "brain"),
    # ("23f48c19-6c88-431c-b518-8d8be7ac8706", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "23f48c19-6c88-431c-b518-8d8be7ac8706", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "brain"),
    # ("6d0d9779-f66b-4297-9bbd-4f232651fb99", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "6d0d9779-f66b-4297-9bbd-4f232651fb99", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "embryonic limb"),
    # ("b23a3e02-d6f9-43a3-bb1b-38af6860c092", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "b23a3e02-d6f9-43a3-bb1b-38af6860c092", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "brain"),
    # ("f92de076-c860-4a4f-9d36-6a74cee3ae5d", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "f92de076-c860-4a4f-9d36-6a74cee3ae5d", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "fallopian tube"),
    # ("64175889-d600-4b58-97ea-e74be80206e5", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "64175889-d600-4b58-97ea-e74be80206e5", "adata_subsample.h5ad", "donor_id", "%", "cell_type", 'retina'),
    # ("484dbc33-c7dc-4e5e-9954-7f2a1cc849bc", "adata.h5ad", "donor_id", "%", "cell_type", "484dbc33-c7dc-4e5e-9954-7f2a1cc849bc", "adata.h5ad", "donor_id", "%", "cell_type", "brain"),
    # ("multiple_sclerosis_scgpt", "c_data.h5ad", "-", "*", "celltype", "multiple_sclerosis_scgpt", "filtered_ms_adata.h5ad", "-", "*", "celltype", "brain (ms lesions)"),
    # ("hPancreas_scgpt", "adata_ref.h5ad", "-", "*", "Celltype", "hPancreas_scgpt", "adata_query.h5ad", "-", "*", "Celltype", "pancreas"),
    # ("myeloid_scgpt", "adata_ref.h5ad", "-", "*", "cell_type", "myeloid_scgpt", "adata_query.h5ad", "-", "*", "cell_type", "different tumor tissues"),
    # ("sea-ad", "adata_subsample.h5ad", "donor_id", "%", "Subclass", "sea-ad", "adata_subsample.h5ad", "donor_id", "%", "Subclass", "brain"),
    # ("sea-ad", "adata_merfish_subsample.h5ad", "Donor ID", "%", "Subclass", "sea-ad", "adata_merfish_subsample.h5ad", "Donor ID", "%", "Subclass", "brain"),
    
    # ("neurips_2021_multiome", "adata_gex_xenium_panel.h5ad", "batch", "s1", "cell_type", "neurips_2021_multiome", "adata_gex_xenium_panel.h5ad", "batch", "s3", "cell_type", "blood"),
    ("neurips_2021_multiome", "adata_gex.h5ad", "batch", "s1", "cell_type", "neurips_2021_multiome", "adata_gex_xenium_panel.h5ad", "batch", "s3", "cell_type", "blood"),
    # ("neurips_2021_multiome", "adata_gex.h5ad", "batch", "*", "cell_type", "neurips_2021_multiome", "adata_gex_xenium_panel.h5ad", "batch", "*", "cell_type", "blood"),
     
    # ("cxg_1M_subset", "train_balanced_2000.h5ad", "-", "*", "cell_type", "4e6932db-5a78-40e4-b961-f87f66ba139a", "adata.h5ad", "-", "*", "cell_type", "blood"),
    # ("cxg_1M_subset", "train_balanced_2000.h5ad", "-", "*", "cell_type", "70bcd657-48e6-405f-9c89-1de62cf16c3e", "adata_subsample.h5ad", "-", "*", "cell_type", "retina"),
    # ("cxg_1M_subset", "train_balanced_2000.h5ad", "-", "*", "cell_type", "8e394b2a-4682-44de-969e-d3fdc5748eda", "adata_subsample.h5ad", "-", "*", "cell_type", "blood"),
    # ("cxg_1M_subset", "train_balanced_2000.h5ad", "-", "*", "cell_type", "d20c92a0-69f8-4612-838c-22f185878ea4", "adata_subsample.h5ad", "-", "*", "cell_type", "blood"),
    
    # ("sea-ad", "adata_subsample.h5ad", "-", "*", "Subclass", "sea-ad", "adata_merfish_subsample.h5ad", "-", "*", "Subclass", "brain"),
    # ("sea-ad", "adata_subsample.h5ad", "-", "*", "Subclass", "sea-ad", "adata_subsample_Merfish_140_panel.h5ad", "-", "*", "Subclass", "brain"),
    # ("sea-ad", "adata_subsample_Merfish_140_panel.h5ad", "-", "*", "Subclass", "sea-ad", "adata_merfish_subsample.h5ad", "-", "*", "Subclass", "brain"),
    # ("71dedd43-6628-4985-80f7-f7784b37abc8", "adata_subsample.h5ad", "-", "*", "cell_type", "71dedd43-6628-4985-80f7-f7784b37abc8", "adata_spatial_artery.h5ad", "-", "*", "cell_type", "artery"),
    # ("10x_scRNA_ovarian_cancer", "adata.h5ad", "-", "*", "cell_type", "spatial_xenium_human_ovarian_cancer_prime_5k_panel_ffpe", "adata_subsample.h5ad", "-", "*", "cell_type", "artery"),
]

results = []

for dataset_config in dataset_configs:
    config = "_".join([f'[{c}]' for c in dataset_config])
    dataset_ref, filename_ref, split_key_ref, split_value_ref, label_key_ref, dataset_query, filename_query, split_key_query, split_value_query, label_key_query, tissue = dataset_config
    print(f"config: {dataset_config}")

    adata_path_ref = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_ref}/h5ads/{filename_ref}"
    adata_path_query = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_query}/h5ads/{filename_query}"
    
    emb_paths = [
        # ("scConcept", "cosine", 0.0005, "6madwcoy/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "6madwcoy/1000/steps/step=20000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "6madwcoy/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "6madwcoy/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "6madwcoy/1000/steps/step=200000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "6madwcoy/1000/steps/step=500000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "6madwcoy/1000/steps/step=1000000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "6madwcoy/1000/steps/step=2000000.ckpt/{}/cell_embs_cls.npy"),
        
        # ("scConcept", "cosine", 0.0005, "6madwcoy/5000/steps/step=1000000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "cypscwzb/5000/steps/step=1000000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "mos5rg7a/4096/steps/step=2000000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "662kfaht/1000/steps/step=2000000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "662kfaht/1000/steps/step=3000000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "e5zbffs2/1000/steps/step=1000000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "z2htg00d/1000/steps/step=1500000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "pgl7wuoq/1000/steps/step=500000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "l668bst9/1000/steps/step=800000.ckpt/{}/cell_embs_cls.npy"),
        
        # within-group-sampling = null
        # ("scConcept", "cosine", 0.0005, "hrz5ufwb/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "hrz5ufwb/1000/steps/step=20000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "hrz5ufwb/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "hrz5ufwb/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "hrz5ufwb/1000/steps/step=200000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "hrz5ufwb/1000/steps/step=500000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept", "cosine", 0.0005, "hrz5ufwb/1000/steps/step=1000000.ckpt/{}/cell_embs_cls.npy"),
        
        # mlm
        # ("scConcept", "cosine", 0.0005, "dvb2g6a1/1000/steps/step=10000.ckpt/{}/cell_embs_mean.npy"),
        # ("scConcept", "cosine", 0.0005, "dvb2g6a1/1000/steps/step=20000.ckpt/{}/cell_embs_mean.npy"),
        # ("scConcept", "cosine", 0.0005, "dvb2g6a1/1000/steps/step=50000.ckpt/{}/cell_embs_mean.npy"),
        # ("scConcept", "cosine", 0.0005, "dvb2g6a1/1000/steps/step=100000.ckpt/{}/cell_embs_mean.npy"),
        # ("scConcept", "cosine", 0.0005, "dvb2g6a1/1000/steps/step=200000.ckpt/{}/cell_embs_mean.npy"),
        # ("scConcept", "cosine", 0.0005, "dvb2g6a1/1000/steps/step=500000.ckpt/{}/cell_embs_mean.npy"),
        
        # scConcept+
        # ("scConcept+", "cosine", 0.0005, "tszen2pq/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "tszen2pq/1000/steps/step=30000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "tszen2pq/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "tszen2pq/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "pztcjbuu/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "pztcjbuu/1000/steps/step=30000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "pztcjbuu/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "pztcjbuu/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "l43bx1dl/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "l43bx1dl/1000/steps/step=30000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "l43bx1dl/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "l43bx1dl/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "k5om4aqx/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "k5om4aqx/1000/steps/step=30000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "k5om4aqx/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "k5om4aqx/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "de9entxo/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "de9entxo/1000/steps/step=30000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "de9entxo/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "de9entxo/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "k7p59riy/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "k7p59riy/1000/steps/step=30000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "k7p59riy/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "k7p59riy/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),


        # ("scConcept+", "cosine", 0.0005, "vcypwbpc/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "vcypwbpc/1000/steps/step=20000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "vcypwbpc/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "vcypwbpc/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),

        # ("scConcept+", "cosine", 0.0005, "meoivn66/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "meoivn66/1000/steps/step=20000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "meoivn66/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "meoivn66/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),

        # ("scConcept+", "cosine", 0.0005, "j2rbbon6/1000/steps/step=10000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "j2rbbon6/1000/steps/step=20000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "j2rbbon6/1000/steps/step=50000.ckpt/{}/cell_embs_cls.npy"),
        # ("scConcept+", "cosine", 0.0005, "j2rbbon6/1000/steps/step=100000.ckpt/{}/cell_embs_cls.npy"),

        # ("PCA", "euclidean", 0.0005, "pca_32_[totalcount_log1p]_[{}]_[{}]/{}/embs.npy"),
        # ("PCA", "euclidean", 0.0005, "pca_128_[totalcount_log1p]_[{}]_[{}]/{}/embs.npy"),
        # ("PCA", "euclidean", 0.0005, "pca_512_[totalcount_log1p]_[{}]_[{}]/{}/embs.npy"),
        
        # ("Outer PCA-128 totalcount_log1p", "euclidean", 0.0005, ""),
        # ("Inner PCA-128 totalcount_log1p", "euclidean", 0.0005, ""),
        
        # ("Inner PCA-32 totalcount_log1p", "euclidean", 0.0005, ""),
        # ("Inner PCA-128 totalcount_log1p", "euclidean", 0.0005, ""),
        # ("Inner PCA-512 totalcount_log1p", "euclidean", 0.0005, ""),        

        # ("Raw Count Inner totalcount_log1p", "euclidean", 0.0005, ""),
        # ("Raw Count Outer totalcount_log1p", "euclidean", 0.0005, ""),
        
        # ("scGPT", "euclidean", 0.005, "scgpt/{}/embs.npy"),
        # ("scGPT-Spatial", "euclidean", 0.005, "scgpt_spatial/{}/embs.npy"),
        # ("Geneformer", "euclidean", 0.005, "geneformer/{}/embs.npy"),
        # ("Nicheformer", "euclidean", 0.0001, "nicheformer/{}/embs.npy"),
    ]

    classifiers = ["knn"]
    min_count = 50
    train_size=None

    for classifier in classifiers:
        for method, metric, lr, emb_path in emb_paths:
            adata_train = sc.read_h5ad(adata_path_ref)
            adata_test = sc.read_h5ad(adata_path_query)
            
            if emb_path != "":
                if "PCA" in method or "RAW" in method:
                    emb_path = emb_path.format(dataset_ref, filename_ref, "{}")
                print(f"Embedding: {emb_path}, lr={lr}, metric={metric}")
                emb_path_ref = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_ref}/embs/{emb_path.format(filename_ref)}"
                emb_path_query = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_query}/embs/{emb_path.format(filename_query)}"

                # assert os.path.exists(emb_path_ref) and os.path.exists(emb_path_query), f"File not found: {emb_path_ref}"
                if not os.path.exists(emb_path_ref) or not os.path.exists(emb_path_query):
                    print(f"File not found!!!!, skipping")
                    continue

                adata_train.obsm['X_emb'] = np.load(emb_path_ref)
                adata_test.obsm['X_emb'] = np.load(emb_path_query)
            
            
            # Check if the entry already exists
            entry_exists = results_df[(results_df.config==config) & 
                                      (results_df.classifier==classifier) & 
                                      (results_df.method==method) & 
                                      (results_df.emb_path==emb_path)
                                      ].shape[0] > 0
            if entry_exists:
                print("Entry exists, skipping")
                continue

    
            # remove low frequency cell types
            freq = adata_train.obs[label_key_ref].value_counts(normalize=False)
            print(f'removing {list(freq[freq < min_count].index)} from dataset')
            adata_train = adata_train[adata_train.obs[label_key_ref].isin(freq[freq > min_count].index)]
            adata_test = adata_test[adata_test.obs[label_key_query].isin(freq[freq > min_count].index)]
                
            adata_train, adata_test = split_train_test(adata_train, adata_test, split_key_ref, split_key_query, split_value_ref, split_value_query)
            
            adata_train = adata_train.copy()
            adata_test = adata_test.copy()
            if metric == 'cosine':
                adata_train.obsm['X_emb'] = F.normalize(torch.tensor(adata_train.obsm['X_emb']), p=2, dim=1).numpy()
                adata_test.obsm['X_emb'] = F.normalize(torch.tensor(adata_test.obsm['X_emb']), p=2, dim=1).numpy()

            if ("PCA" in method or "RAW" in method) and emb_path == "":
                print(f"Embedding: {method}, lr={lr}, metric={metric}")
                adata_train, adata_test = extract_baseline_embs(adata_train, adata_test, method, key='X_emb')
                
            train_embs = adata_train.obsm['X_emb']
            test_embs = adata_test.obsm['X_emb']
            train_labels = adata_train.obs[label_key_ref]
            test_labels = adata_test.obs[label_key_query]
            
            if train_size is not None:
                # adata_balanced = balance_anndata(adata_train, min_cells_per_class, label_key)
                sc.pp.subsample(adata_train, n_obs=train_size, copy=False, random_state=42)
                # adata_train = ad.concat([adata_train, adata_balanced], axis=0)

            accuracy, f1_score_weighted, f1_score_macro, y_pred = evaluate(train_embs, train_labels, test_embs, test_labels, classifier, metric=metric, pred_within_test_labels=False, lr=lr)
            print(f'{accuracy:.3f}, {f1_score_weighted:.3f}, {f1_score_macro:.3f}')

            
            results.append((config, tissue, dataset_ref, filename_ref, dataset_query, filename_query, classifier, method, emb_path, accuracy, f1_score_weighted, f1_score_macro))


config: ('neurips_2021_multiome', 'adata_gex.h5ad', 'batch', 's1', 'cell_type', 'neurips_2021_multiome', 'adata_gex_xenium_panel.h5ad', 'batch', 's3', 'cell_type', 'blood')
removing [] from dataset
Embedding: Outer PCA-128 totalcount_log1p, lr=0.0005, metric=euclidean
Normalization: totalcount_log1p


/ictstr01/groups/ml01/workspace/mojtaba.bahrami/projects/contrastive-transformer-reproducibility/ct_rep/utils/utils.py:73: ImplicitModificationWarning: Setting element `.obsm['X_emb']` of view, initializing view as actual.
  adata_query.obsm[key] = pca.transform(X_query)


train split: 15560 cells containing 22 cell types, embeddings of size 128
test split: 13270 cells containing 21 cell types, embeddings of size 128
Fitting KNN (euclidean) with n_neighbors:  10
knn Accuracy: 0.018
knn F1-Weighted: 0.001
knn F1-Macro: 0.003
0.018, 0.001, 0.003
removing [] from dataset
Embedding: Inner PCA-128 totalcount_log1p, lr=0.0005, metric=euclidean
Normalization: totalcount_log1p
train split: 15560 cells containing 22 cell types, embeddings of size 128
test split: 13270 cells containing 21 cell types, embeddings of size 128
Fitting KNN (euclidean) with n_neighbors:  10
knn Accuracy: 0.510
knn F1-Weighted: 0.509
knn F1-Macro: 0.377
0.510, 0.509, 0.377


In [19]:
results_df = pd.concat([results_df, pd.DataFrame(results, columns=columns)])

duplicates = results_df.duplicated(subset=results_df.columns[:-3], keep='last')
print(f"Removing {duplicates.sum()} duplicates")
results_df = results_df[~duplicates]

Removing 0 duplicates


In [52]:
results_df.to_csv(results_path, index=False)

# Raw count / PCA

First select the dataset config and embedding from the previous section

In [31]:
# dataset_config = ("neurips_2021_multiome", "adata_gex.h5ad", "batch", "s1", "cell_type", "neurips_2021_multiome", "adata_gex.h5ad", "batch", "s3", "cell_type", "blood")
dataset_config = ("neurips_2021_multiome", "adata_gex.h5ad", "batch", "s1", "cell_type", "neurips_2021_multiome", "adata_gex_xenium_panel.h5ad", "batch", "s3", "cell_type", "blood")
# dataset_config = ("sea-ad", "adata_subsample.h5ad", "-", "*", "Subclass", "sea-ad", "adata_merfish_subsample.h5ad", "-", "*", "Subclass", "brain")
# dataset_config = ("10x_scRNA_ovarian_cancer", "adata.h5ad", "-", "*", "cell_type", "spatial_xenium_human_ovarian_cancer_prime_5k_panel_ffpe", "adata_subsample.h5ad", "-", "*", "cell_type", "artery")

config = "_".join([f'[{c}]' for c in dataset_config])
dataset_ref, filename_ref, split_key_ref, split_value_ref, label_key_ref, dataset_query, filename_query, split_key_query, split_value_query, label_key_query, tissue = dataset_config

adata_path_ref = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_ref}/h5ads/{filename_ref}"
adata_path_query = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_query}/h5ads/{filename_query}"

In [32]:
method = "Inner PCA-128 totalcount_log1p" # Inner PCA-128, Raw Count Inner
emb_path = "Inner PCA-128 totalcount_log1p" # Inner PCA-128, Raw Count Inner
normalization = "totalcount_log1p" # raw, log1p, totalcount_log1p
classifier = 'knn' # knn, linear
metric = 'euclidean'
lr = 0.0005
min_count = 50

In [ ]:
adata_train = sc.read_h5ad(adata_path_ref)
adata_test = sc.read_h5ad(adata_path_query)
# adata_train.obsm['X_emb'] = np.load(emb_path_ref)
# adata_test.obsm['X_emb'] = np.load(emb_path_query)

# remove low frequency cell types
freq = adata_train.obs[label_key_ref].value_counts(normalize=False)
print(f'removing {list(freq[freq < min_count].index)} from dataset')
adata_train = adata_train[adata_train.obs[label_key_ref].isin(freq[freq > min_count].index)]
adata_test = adata_test[adata_test.obs[label_key_query].isin(freq[freq > min_count].index)]

# split
adata_train, adata_test = split_train_test(adata_train, adata_test, split_key_ref, split_key_query, split_value_ref, split_value_query)

adata_train = adata_train.copy()
adata_test = adata_test.copy()

############################################################################################################

adata_train, adata_test = extract_baseline_embs(adata_train, adata_test, method, key='X_emb')

############################################################################################################
# if metric == 'cosine':
#     adata_train.obsm['X_emb'] = F.normalize(torch.tensor(adata_train.obsm['X_emb']), p=2, dim=1).numpy()
#     adata_test.obsm['X_emb'] = F.normalize(torch.tensor(adata_test.obsm['X_emb']), p=2, dim=1).numpy()

train_embs = adata_train.obsm['X_emb']
test_embs = adata_test.obsm['X_emb']

train_labels = adata_train.obs[label_key_ref]
test_labels = adata_test.obs[label_key_query]

if train_size is not None:
    # adata_balanced = balance_anndata(adata_train, min_cells_per_class, label_key)
    sc.pp.subsample(adata_train, n_obs=train_size, copy=False, random_state=42)
    # adata_train = ad.concat([adata_train, adata_balanced], axis=0)

accuracy, f1_score_weighted, f1_score_macro, y_pred = evaluate(train_embs, train_labels, test_embs, test_labels, classifier, metric=metric, pred_within_test_labels=False, lr=lr)
print(f'{accuracy:.3f}, {f1_score_weighted:.3f}, {f1_score_macro:.3f}')

removing [] from dataset
Normalization: totalcount_log1p
train split: 15560 cells containing 22 cell types, embeddings of size 128
test split: 13270 cells containing 21 cell types, embeddings of size 128
Fitting KNN (euclidean) with n_neighbors:  10
knn Accuracy: 0.510
knn F1-Weighted: 0.509
knn F1-Macro: 0.377
0.510, 0.509, 0.377


In [29]:
results = []

In [30]:
results.append((config, tissue, dataset_ref, filename_ref, dataset_query, filename_query, classifier, f'{method} {normalization}', f'{method} {normalization}', accuracy, f1_score_weighted, f1_score_macro))

In [31]:
len(results)

1

In [32]:
results_df = pd.concat([results_df, pd.DataFrame(results, columns=columns)])

duplicates = results_df.duplicated(subset=results_df.columns[:-3], keep='last')
print(f"Removing {duplicates.sum()} duplicates")
results_df = results_df[~duplicates]

Removing 0 duplicates


In [33]:
results_df.to_csv(results_path, index=False)

# Celltypist

First select the dataset config and embedding from the previous section

In [78]:
from ct_rep.celltype.celltype_pred import evaluate
from ct_rep.utils.utils import sample_data, adapt_vars
from ct_rep.utils.utils import normalize
import celltypist

In [ ]:
dataset_config = ("neurips_2021_multiome", "adata_gex.h5ad", "batch", "s1", "cell_type", "neurips_2021_multiome", "adata_gex.h5ad", "batch", "s3", "cell_type", "blood")
# dataset_config = ("7840398e-2e12-49d2-b21a-f59a1908057e", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "7840398e-2e12-49d2-b21a-f59a1908057e", "adata_subsample.h5ad", "donor_id", "%", "cell_type", "skeletal muscle")
# dataset_config = ("multiple_sclerosis_scgpt", "c_data.h5ad", "-", "*", "celltype", "multiple_sclerosis_scgpt", "filtered_ms_adata.h5ad", "-", "*", "celltype", "brain (ms lesions)")

config = "_".join([f'[{c}]' for c in dataset_config])
dataset_ref, filename_ref, split_key_ref, split_value_ref, label_key_ref, dataset_query, filename_query, split_key_query, split_value_query, label_key_query, tissue = dataset_config

adata_path_ref = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_ref}/h5ads/{filename_ref}"
adata_path_query = f"/lustre/groups/ml01/projects/contrastive_transformer/data/{dataset_query}/h5ads/{filename_query}"

In [79]:
method = "CellTypist" # Inner PCA-32, Raw Count Inner
emb_path = "CellTypist" # Inner PCA-32, Raw Count Inner
normalization = "totalcount/log1p" # raw, log1p, totalcount/log1p
classifier = 'linear'
metric = 'euclidean'
min_count = 50
train_size=None

In [80]:
adata_train = sc.read_h5ad(adata_path_ref)
adata_test = sc.read_h5ad(adata_path_query)

# remove low frequency cell types
freq = adata_train.obs[label_key_ref].value_counts(normalize=False)
print(f'removing {list(freq[freq < min_count].index)} from dataset')
adata_train = adata_train[adata_train.obs[label_key_ref].isin(freq[freq > min_count].index)]
adata_test = adata_test[adata_test.obs[label_key_query].isin(freq[freq > min_count].index)]


# split
if split_value_ref=="%":
    assert (adata_train.obs.index == adata_test.obs.index).all(), 'index mismatch'
    train_split, test_split = train_test_split(list(adata_train.obs[split_key_ref].unique()), test_size=0.5, random_state=42)
    print(f'train split: {train_split}, test split: {test_split}')
    adata_train = adata_train[adata_train.obs[split_key_ref].isin(train_split)]
    adata_test = adata_test[adata_test.obs[split_key_query].isin(test_split)]
elif split_value_ref=="*":
    adata_train = adata_train
    adata_test = adata_test
else:
    adata_train = adata_train[adata_train.obs[split_key_ref].str.contains(split_value_ref)]
    adata_test = adata_test[adata_test.obs[split_key_query].str.contains(split_value_query)]


adata_train = adata_train.copy()
adata_test = adata_test.copy()
############################################################################################################
shared_genes = adata_train.var.index.intersection(adata_test.var.index)
adata_train = adata_train[:, shared_genes].copy()
adata_test = adata_test[:, shared_genes].copy()

X_ref = normalize(adata_train.X.toarray(), normalization)
X_query = normalize(adata_test.X.toarray(), normalization)    
############################################################################################################
adata_train.X = X_ref
adata_test.X = X_query

train_labels = adata_train.obs[label_key_ref]
test_labels = adata_test.obs[label_key_query]

if train_size is not None:
    # adata_balanced = balance_anndata(adata_train, min_cells_per_class, label_key)
    sc.pp.subsample(adata_train, n_obs=train_size, copy=False, random_state=42)
    # adata_train = ad.concat([adata_train, adata_balanced], axis=0)

removing [] from dataset


In [ ]:
ct_model = celltypist.train(
    adata_train,
    labels=train_labels, 
    n_jobs=10, 
    feature_selection=True,
    use_SGD=True, 
    use_GPU=True,
    mini_batch=True,
    # batch_number=256,
    with_mean=False,
    random_state=1,
)

🍳 Preparing data before training
🔬 Input data has 15560 cells and 13431 genes
⚖️ Scaling input data
🏋️ Training data using mini-batch SGD logistic regression
⏳ Epochs: [1/10]
⏳ Epochs: [2/10]
⏳ Epochs: [3/10]
⏳ Epochs: [4/10]
⏳ Epochs: [5/10]
⏳ Epochs: [6/10]
⏳ Epochs: [7/10]
⏳ Epochs: [8/10]
⏳ Epochs: [9/10]
⏳ Epochs: [10/10]
🔎 Selecting features
🧬 4763 features are selected
🏋️ Starting the second round of training
🏋️ Training data using mini-batch SGD logistic regression
⏳ Epochs: [1/10]
⏳ Epochs: [2/10]
⏳ Epochs: [3/10]
⏳ Epochs: [4/10]
⏳ Epochs: [5/10]
⏳ Epochs: [6/10]
⏳ Epochs: [7/10]
⏳ Epochs: [8/10]
⏳ Epochs: [9/10]
⏳ Epochs: [10/10]
✅ Model training done!


In [83]:
preds = celltypist.annotate(adata_test, model=ct_model) # majority_voting = True

🔬 Input data has 13270 cells and 13431 genes
🔗 Matching reference genes in the model
🧬 4763 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!


In [84]:
y_pred = preds.predicted_labels.predicted_labels

In [ ]:
accuracy = accuracy_score(test_labels, y_pred)
f1_score_weighted = f1_score(test_labels, y_pred, labels=np.unique(test_labels), average='weighted')
f1_score_macro = f1_score(test_labels, y_pred, labels=np.unique(test_labels), average='macro')
print(f'{classifier} Accuracy: {accuracy:.3f}')
print(f'{classifier} F1-Weighted: {f1_score_weighted:.3f}')
print(f'{classifier} F1-Macro: {f1_score_macro:.3f}')

linear Accuracy: 0.715
linear F1-Weighted: 0.706
linear F1-Macro: 0.637


In [86]:
results = []
results.append((config, tissue, dataset_ref, filename_ref, dataset_query, filename_query, classifier, method, method, accuracy, f1_score_weighted, f1_score_macro))

In [87]:
len(results)

1

In [ ]:
results_df = pd.concat([results_df, pd.DataFrame(results, columns=columns)])

duplicates = results_df.duplicated(subset=results_df.columns[:-3], keep='last')
print(f"Removing {duplicates.sum()} duplicates")
results_df = results_df[~duplicates]

Removing 1 duplicates


In [69]:
results_df.to_csv(results_path, index=False)